In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_ENABLE_X64"] = "true"

import numpy as onp
import numpy.typing as npt
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from tqdm import tqdm
from typing import Callable, NamedTuple

from msmjax.bspline_basis import create_bspline_basis_element

from gridops import set_up_grid_axis
from gridops import create_anterpolation_function

In [2]:
p = 4
order =  p - 1

# Grid charge computation

In [10]:
length = 10.
h = 0.25
n_particles = 25

grid = set_up_grid_axis(length=length, h=h, p=p, periodic=False)

anterpolate = create_anterpolation_function(grid)
jitted_anterpolate = jax.jit(anterpolate)

In [11]:
rng = onp.random.default_rng(12345)
for _ in tqdm(range(100)):
    positions = jnp.array(rng.uniform(0.0, length, size=n_particles))
    charges = jnp.array(rng.uniform(-1., 1., size=n_particles))
    
    # Call to trigger compilation
    gridcharge, _, _ = jitted_anterpolate(positions, charges)
    # Second call
    gridcharge, _, _ = jitted_anterpolate(positions, charges)
    
    assert jnp.allclose(gridcharge.sum(), charges.sum())

100%|██████████| 100/100 [00:00<00:00, 460.84it/s]


In [13]:
gridcharge

Array([ 0.00000000e+00, -2.20895703e-02, -7.21955552e-02,  4.26507768e-01,
        3.29568189e-01,  7.23824047e-03,  2.32713530e-04,  1.03667146e-01,
        2.66392054e-01,  4.21979956e-02,  6.74068875e-03,  3.30153620e-01,
        4.79153039e-01, -3.06775366e-01, -2.94411676e-01, -7.74144703e-02,
       -1.09600666e-01, -1.56720723e-01, -3.66969234e-01, -2.74730081e-01,
       -3.35428598e-02, -2.99325842e-02, -2.56905306e-03, -1.49392044e-02,
       -2.21182511e-01, -1.69071643e-01, -4.28063705e-03, -1.13500813e-03,
       -5.80471872e-02, -4.15888721e-01, -5.20213559e-01,  1.03357089e-01,
        2.07316078e-01, -1.67594088e-01, -6.81575186e-01, -1.51845703e-01,
        0.00000000e+00,  3.62461835e-03,  1.75952112e-01,  2.55234466e-01,
        2.91401722e-01,  5.84573301e-01,  1.23826059e-01,  4.11681290e-04],      dtype=float64)